In [1]:
import torch
import subprocess

print(
    subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    ).stdout
)

if torch.cuda.is_available():

    free, total = torch.cuda.mem_get_info()

    print(
        f"CUDA free : {free / 1024**3:.2f} GB"
    )

    print(
        f"CUDA total: {total / 1024**3:.2f} GB"
    )

[HAMI-core Msg(1491:140683470724416:libvgpu.c:839)]: Initializing.....


Mon Aug 17 09:55:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:C2:00.0 Off |                    0 |
| N/A   49C    P0            102W /  300W |     477MiB /  16384MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

[HAMI-core Msg(1491:140683470724416:libvgpu.c:855)]: Initialized


In [1]:
# ============================================================
# QC-FILTERED 1x AUGMENTED XLM-R EXPERIMENT
# CANONICAL MATCHED CONFIGURATION
#
# IMPORTANT:
# This configuration is matched to the FINAL ORIGINAL
# RUHSOLD XLM-R baseline and the corrected unfiltered run.
#
# Intended experimental difference:
# TRAINING DATA ONLY
# ============================================================

from pathlib import Path

import os
import gc
import math
import random
import time
import shutil

import numpy as np
import pandas as pd
import torch


# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)


# ============================================================
# ORIGINAL RUHSOLD PATHS
# ============================================================

TRAIN_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_train.tsv"
)

VAL_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_validation.tsv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_test.tsv"
)


# ============================================================
# QC-FILTERED SYNTHETIC BANK
# ============================================================

SYNTHETIC_BANK_PATH = (
    PROJECT_ROOT
    / "fine tuning"
    / "outputs"
    / "synthetic_generation"
    / "qc_pipeline"
    / "full_production"
    / "accepted_synthetic_bank.csv"
)


# ============================================================
# CORRECTED OUTPUT DIRECTORIES
#
# NEW directories are used so the corrected experiment does
# not mix with the earlier direct-BF16 augmented run.
# ============================================================

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "classifier"
    / "outputs"
    / "xlm_roberta_qc_filtered_1x_corrected"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "classifier"
    / "outputs"
    / "xlm_roberta_qc_filtered_1x_corrected_results"
)


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_NAME = (
    "FacebookAI/xlm-roberta-base"
)

NUM_LABELS = 5

MAX_LENGTH = 128


# ============================================================
# RUHSOLD LABEL MAPPING
# ============================================================

id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

label2id = {
    label: class_id
    for class_id, label
    in id2label.items()
}


# ============================================================
# FROZEN OPTUNA-SELECTED HYPERPARAMETERS
#
# EXACTLY THE SAME VALUES AS:
# - final original XLM-R baseline
# - corrected unfiltered 1x experiment
#
# No retuning is performed.
# ============================================================

BEST_LEARNING_RATE = (
    2.881129057462248e-05
)

BEST_WEIGHT_DECAY = (
    0.03348739395854274
)

BEST_WARMUP_RATIO = (
    0.03756172525242821
)

FINAL_MAX_EPOCHS = 10

FINAL_EARLY_STOPPING_PATIENCE = 2


# ============================================================
# CANONICAL BATCH CONFIGURATION
#
# EXACT MATCH TO FINAL ORIGINAL + CORRECTED UNFILTERED:
#
# physical train batch = 16
# gradient accumulation = 1
# effective train batch = 16
# evaluation batch      = 16
# ============================================================

TRAIN_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 1

EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

EVAL_BATCH_SIZE = 16


assert (
    EFFECTIVE_BATCH_SIZE
    ==
    16
)


# Backward-compatible name if later cells still use it.
BEST_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
)


# ============================================================
# REPEATED-RUN SEEDS
# ============================================================

SEEDS = [
    42,
    43,
    44,
]


# ============================================================
# EXPECTED DATASET SIZES
#
# QC-filtered 1x should use:
#
# Original RUHSOLD = 6408
# Synthetic        = 1448
# Final            = 7856
# ============================================================

ORIGINAL_TRAIN_SIZE = 6408

SYNTHETIC_1X_SIZE = 1448

FINAL_FILTERED_TRAIN_SIZE = (
    ORIGINAL_TRAIN_SIZE
    +
    SYNTHETIC_1X_SIZE
)


assert (
    FINAL_FILTERED_TRAIN_SIZE
    ==
    7856
)


# ============================================================
# WARMUP CONVERSION
#
# Same frozen warmup RATIO as the original baseline.
#
# 7856 / 16
# = 491 optimizer steps per epoch
#
# 491 x 10
# = 4910 maximum optimizer steps
#
# ceil(
#     4910 x 0.03756172525242821
# )
# = 185 warmup steps
# ============================================================

MINI_BATCHES_PER_EPOCH = math.ceil(
    FINAL_FILTERED_TRAIN_SIZE
    / TRAIN_BATCH_SIZE
)


OPTIMIZER_STEPS_PER_EPOCH = math.ceil(
    MINI_BATCHES_PER_EPOCH
    / GRADIENT_ACCUMULATION_STEPS
)


MAX_OPTIMIZER_STEPS = (
    OPTIMIZER_STEPS_PER_EPOCH
    * FINAL_MAX_EPOCHS
)


BEST_WARMUP_STEPS = math.ceil(
    MAX_OPTIMIZER_STEPS
    * BEST_WARMUP_RATIO
)


assert (
    MINI_BATCHES_PER_EPOCH
    ==
    491
)

assert (
    OPTIMIZER_STEPS_PER_EPOCH
    ==
    491
)

assert (
    MAX_OPTIMIZER_STEPS
    ==
    4910
)

assert (
    BEST_WARMUP_STEPS
    ==
    185
)


# ============================================================
# CHECKPOINT SELECTION METRIC
# ============================================================

CHECKPOINT_SELECTION_METRIC = (
    "macro_f1"
)


# ============================================================
# METRIC LABEL ORDER
# ============================================================

METRIC_LABELS = [
    0,
    1,
    2,
    3,
    4,
]


# ============================================================
# BASE REPRODUCIBILITY
# ============================================================

GLOBAL_SEED = 42


random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)

torch.manual_seed(
    GLOBAL_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        GLOBAL_SEED
    )


# ============================================================
# CONFIGURATION VERIFICATION
# ============================================================

print("=" * 80)

print(
    "QC-FILTERED 1x AUGMENTED XLM-R "
    "CANONICAL MATCHED CONFIGURATION"
)

print("=" * 80)


print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


print(
    "\nModel:",
    MODEL_NAME
)

print(
    "Seeds:",
    SEEDS
)


print(
    "\nFrozen tuned hyperparameters:"
)

print(
    "Learning rate:",
    BEST_LEARNING_RATE
)

print(
    "Weight decay:",
    BEST_WEIGHT_DECAY
)

print(
    "Warmup ratio:",
    BEST_WARMUP_RATIO
)


print(
    "\nCanonical batch configuration:"
)

print(
    "Physical train batch:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch:",
    EVAL_BATCH_SIZE
)


print(
    "\nExpected training sizes:"
)

print(
    "Original training samples:",
    ORIGINAL_TRAIN_SIZE
)

print(
    "QC-filtered synthetic samples:",
    SYNTHETIC_1X_SIZE
)

print(
    "Final augmented samples:",
    FINAL_FILTERED_TRAIN_SIZE
)


print(
    "\nTraining schedule:"
)

print(
    "Optimizer steps per epoch:",
    OPTIMIZER_STEPS_PER_EPOCH
)

print(
    "Maximum optimizer steps:",
    MAX_OPTIMIZER_STEPS
)

print(
    "Equivalent warmup steps:",
    BEST_WARMUP_STEPS
)


print(
    "\nMaximum epochs:",
    FINAL_MAX_EPOCHS
)

print(
    "Early stopping patience:",
    FINAL_EARLY_STOPPING_PATIENCE
)

print(
    "Checkpoint selection metric:",
    CHECKPOINT_SELECTION_METRIC
)


print(
    "\nSynthetic bank exists:",
    SYNTHETIC_BANK_PATH.exists()
)

print(
    SYNTHETIC_BANK_PATH
)


print(
    "\nCorrected output root:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nCorrected results root:"
)

print(
    RESULTS_ROOT
)


print(
    "\nQC-FILTERED CANONICAL CONFIGURATION VERIFIED."
)

[HAMI-core Msg(162:140702996176192:libvgpu.c:839)]: Initializing.....


QC-FILTERED 1x AUGMENTED XLM-R CANONICAL MATCHED CONFIGURATION
CUDA available: True
GPU: NVIDIA L40S

Model: FacebookAI/xlm-roberta-base
Seeds: [42, 43, 44]

Frozen tuned hyperparameters:
Learning rate: 2.881129057462248e-05
Weight decay: 0.03348739395854274
Warmup ratio: 0.03756172525242821

Canonical batch configuration:
Physical train batch: 16
Gradient accumulation: 1
Effective batch: 16
Evaluation batch: 16

Expected training sizes:
Original training samples: 6408
QC-filtered synthetic samples: 1448
Final augmented samples: 7856

Training schedule:
Optimizer steps per epoch: 491
Maximum optimizer steps: 4910
Equivalent warmup steps: 185

Maximum epochs: 10
Early stopping patience: 2
Checkpoint selection metric: macro_f1

Synthetic bank exists: True
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/full_production/accepted_synthetic_bank.csv

Corrected output root:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_qc

[HAMI-core Msg(162:140702996176192:libvgpu.c:855)]: Initialized


In [2]:
# ============================================================
# LOAD ORIGINAL RUHSOLD SPLITS
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    names=[
        "tweet",
        "label",
    ]
)

val_df = pd.read_csv(
    VAL_PATH,
    sep="\t",
    names=[
        "tweet",
        "label",
    ]
)

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t",
    names=[
        "tweet",
        "label",
    ]
)

print(
    "Original training samples:",
    len(train_df)
)

print(
    "Validation samples:",
    len(val_df)
)

print(
    "Test samples:",
    len(test_df)
)


# ============================================================
# LOAD QC-PASSED SYNTHETIC BANK
# ============================================================

synthetic_bank_df = pd.read_csv(
    SYNTHETIC_BANK_PATH
)

print(
    "\nQC-passed synthetic samples:",
    len(synthetic_bank_df)
)

print("\nSynthetic class distribution:")

display(
    synthetic_bank_df[
        [
            "class_id",
            "target_label",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)

Original training samples: 6408
Validation samples: 801
Test samples: 2003

QC-passed synthetic samples: 751

Synthetic class distribution:


,class_id,target_label,count
0,2,Religious Hate,279
1,4,Profane,253
2,3,Sexism,219


In [4]:
# ============================================================
# PREPARE SYNTHETIC DATA FOR CLASSIFIER TRAINING
# ============================================================

synthetic_train_df = (
    synthetic_bank_df[
        [
            "generated_text",
            "class_id",
        ]
    ]
    .copy()
)

synthetic_train_df = (
    synthetic_train_df.rename(
        columns={
            "generated_text": "tweet",
            "class_id": "label",
        }
    )
)

synthetic_train_df[
    "tweet"
] = (
    synthetic_train_df[
        "tweet"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

synthetic_train_df[
    "label"
] = (
    synthetic_train_df[
        "label"
    ]
    .astype(int)
)

print(
    "Prepared synthetic training samples:",
    len(synthetic_train_df)
)

display(
    synthetic_train_df.head()
)

Prepared synthetic training samples: 751


,tweet,label
0,bhenchod i wanna cry but can’t 😔,4
1,nhi pata na bhenchod,4
2,bhenchod shayad yeh hai na jisne mera sara lif...,4
3,rt : shuru karne ka sath dala jata hai aur peh...,4
4,rt : bhenchod jinko kuch nhi mil raha wo dm ka...,4


In [5]:
# ============================================================
# CREATE AUGMENTED TRAINING SET
# ============================================================

train_aug_df = pd.concat(
    [
        train_df[
            [
                "tweet",
                "label",
            ]
        ].copy(),

        synthetic_train_df,
    ],
    ignore_index=True,
)

print(
    "Original training size:",
    len(train_df)
)

print(
    "Synthetic additions:",
    len(synthetic_train_df)
)

print(
    "Augmented training size:",
    len(train_aug_df)
)

print("\nOriginal class distribution:")

display(
    train_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

print("\nAugmented class distribution:")

display(
    train_aug_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

Original training size: 6408
Synthetic additions: 751
Augmented training size: 7159

Original class distribution:


,label,count
0,0,1537
1,1,3423
2,2,500
3,3,537
4,4,411



Augmented class distribution:


,label,count
0,0,1537
1,1,3423
2,2,779
3,3,756
4,4,664


In [6]:
# ============================================================
# DATA LEAKAGE SANITY CHECK
# Synthetic examples vs validation/test
# ============================================================

synthetic_text_set = set(
    synthetic_train_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

val_text_set = set(
    val_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

test_text_set = set(
    test_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

synthetic_val_overlap = (
    synthetic_text_set
    &
    val_text_set
)

synthetic_test_overlap = (
    synthetic_text_set
    &
    test_text_set
)

print(
    "Exact synthetic-validation overlap:",
    len(synthetic_val_overlap)
)

print(
    "Exact synthetic-test overlap:",
    len(synthetic_test_overlap)
)

Exact synthetic-validation overlap: 0
Exact synthetic-test overlap: 0


In [21]:
# ============================================================
# EXACT-TEXT LEAKAGE CHECK
# SYNTHETIC TRAINING DATA VS VALIDATION / TEST
# ============================================================

synthetic_text_set = set(
    synthetic_classifier_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

val_text_set = set(
    val_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

test_text_set = set(
    test_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)


synthetic_val_overlap = (
    synthetic_text_set
    &
    val_text_set
)

synthetic_test_overlap = (
    synthetic_text_set
    &
    test_text_set
)


print(
    "Synthetic-validation exact overlap:",
    len(synthetic_val_overlap)
)

print(
    "Synthetic-test exact overlap:",
    len(synthetic_test_overlap)
)


assert (
    len(synthetic_val_overlap)
    == 0
)

assert (
    len(synthetic_test_overlap)
    == 0
)

print(
    "\nNo exact synthetic overlap with validation or test."
)

Synthetic-validation exact overlap: 0
Synthetic-test exact overlap: 0

No exact synthetic overlap with validation or test.


In [4]:
# ============================================================
# RECREATE FINAL 1x AUGMENTED DATASET
# ============================================================

train_aug_dataset = RUHSOLDDataset(
    dataframe=train_aug_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = RUHSOLDDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_dataset = RUHSOLDDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

print(
    "Augmented training dataset:",
    len(train_aug_dataset)
)

print(
    "Validation dataset:",
    len(val_dataset)
)

print(
    "Test dataset:",
    len(test_dataset)
)

NameError: name 'RUHSOLDDataset' is not defined

In [5]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

import random
import numpy as np
import torch


def set_seed(seed: int) -> None:
    """
    Set random seeds across Python, NumPy and PyTorch.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print("Reproducibility function loaded.")

Reproducibility function loaded.


In [6]:
# ============================================================
# LABEL MAPPINGS
# ============================================================

id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

label2id = {
    label: idx
    for idx, label in id2label.items()
}

print("Label mappings loaded.")


# ============================================================
# TOKENIZER
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer:",
    tokenizer.__class__.__name__
)

print(
    "Vocabulary size:",
    tokenizer.vocab_size
)

Label mappings loaded.


Tokenizer: XLMRobertaTokenizer
Vocabulary size: 250002


In [7]:
# ============================================================
# RUHSOLD DATASET CLASS
# ============================================================

from torch.utils.data import Dataset


class RUHSOLDDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length,
    ):

        self.tweets = (
            dataframe["tweet"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["label"]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):

        return len(
            self.tweets
        )

    def __getitem__(
        self,
        index,
    ):

        tweet = self.tweets[
            index
        ]

        label = self.labels[
            index
        ]

        encoded = self.tokenizer(
            tweet,
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )

        encoded["labels"] = label

        return encoded


print(
    "RUHSOLDDataset loaded."
)

RUHSOLDDataset loaded.


In [8]:
# ============================================================
# CREATE AUGMENTED TRAIN + ORIGINAL VALIDATION DATASETS
# ============================================================

train_aug_dataset = RUHSOLDDataset(
    dataframe=train_aug_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = RUHSOLDDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

print(
    "Augmented training dataset:",
    len(train_aug_dataset)
)

print(
    "Original validation dataset:",
    len(val_dataset)
)

Augmented training dataset: 7856
Original validation dataset: 801


In [9]:
# ============================================================
# DYNAMIC BATCH PADDING
# ============================================================

from transformers import (
    DataCollatorWithPadding,
)

data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt",
    )
)

print(
    "Data collator created."
)

Data collator created.


In [10]:
# ============================================================
# EVALUATION METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)


def compute_metrics(
    eval_prediction,
):

    logits, labels = (
        eval_prediction
    )

    predictions = np.argmax(
        logits,
        axis=-1,
    )

    accuracy = accuracy_score(
        labels,
        predictions,
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "weighted_precision":
            weighted_precision,

        "weighted_recall":
            weighted_recall,

        "weighted_f1":
            weighted_f1,
    }


print(
    "Evaluation metrics loaded."
)

Evaluation metrics loaded.


In [31]:
required_objects = [
    "train_aug_df",
    "val_df",
    "test_df",
    "tokenizer",
    "RUHSOLDDataset",
    "train_aug_dataset",
    "val_dataset",
    "test_dataset",
    "data_collator",
    "compute_metrics",
    "set_seed",
]

for name in required_objects:
    print(
        f"{name:<20}",
        "✓" if name in globals() else "✗ MISSING"
    )

train_aug_df         ✓
val_df               ✓
test_df              ✓
tokenizer            ✓
RUHSOLDDataset       ✓
train_aug_dataset    ✓
val_dataset          ✓
test_dataset         ✓
data_collator        ✓
compute_metrics      ✓
set_seed             ✓


In [11]:
# ============================================================
# QC-FILTERED 1x AUGMENTED XLM-R
# LOAD + FREEZE + VERIFY FINAL TRAINING DATA
#
# IMPORTANT:
# The ONLY synthetic dataset used in the final classifier
# experiment is:
#
# final_1x_synthetic_training_set.csv
#
# The augmented dataframe is constructed ONCE and is reused
# unchanged for seeds 42, 43, and 44.
#
# The test set is NOT instantiated here.
# ============================================================

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd


# ============================================================
# FINAL FROZEN QC-FILTERED SYNTHETIC DATASET
# ============================================================

FINAL_SYNTHETIC_PATH = Path(
    "/home/jovyan/project work/data_analyssis/"
    "fine tuning/outputs/synthetic_generation/"
    "qc_pipeline/round2/"
    "final_1x_synthetic_training_set.csv"
)


assert (
    FINAL_SYNTHETIC_PATH.exists()
), (
    f"Final synthetic dataset not found:\n"
    f"{FINAL_SYNTHETIC_PATH}"
)


print("=" * 80)
print(
    "QC-FILTERED 1x SYNTHETIC DATA VERIFICATION"
)
print("=" * 80)

print(
    "Frozen synthetic file:"
)

print(
    FINAL_SYNTHETIC_PATH
)


# ============================================================
# LOAD FINAL FROZEN SYNTHETIC DATASET
# ============================================================

synthetic_df = pd.read_csv(
    FINAL_SYNTHETIC_PATH
)


print(
    "\nRaw synthetic dataset shape:",
    synthetic_df.shape
)

print(
    "Available columns:"
)

print(
    synthetic_df.columns.tolist()
)


# ============================================================
# REQUIRED COLUMNS
# ============================================================

required_synthetic_columns = {
    "generated_text",
    "class_id",
}


assert (
    required_synthetic_columns
    .issubset(
        synthetic_df.columns
    )
), (
    "Required synthetic columns are missing."
)


# ============================================================
# CONVERT TO CLASSIFIER FORMAT
# ============================================================

synthetic_classifier_df = (
    synthetic_df[
        [
            "generated_text",
            "class_id",
        ]
    ]
    .copy()
)


synthetic_classifier_df = (
    synthetic_classifier_df.rename(
        columns={
            "generated_text":
                "tweet",

            "class_id":
                "label",
        }
    )
)


# ============================================================
# NORMALIZE TYPES ONLY
#
# IMPORTANT:
# We are NOT applying semantic/text normalization here.
# This is only datatype cleanup.
# ============================================================

synthetic_classifier_df[
    "tweet"
] = (
    synthetic_classifier_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)


synthetic_classifier_df[
    "label"
] = (
    synthetic_classifier_df[
        "label"
    ]
    .astype(int)
)


# ============================================================
# VERIFY NO EMPTY / MISSING SYNTHETIC TEXT
# ============================================================

assert (
    synthetic_classifier_df[
        "tweet"
    ]
    .isna()
    .sum()
    ==
    0
)


assert (
    synthetic_classifier_df[
        "label"
    ]
    .isna()
    .sum()
    ==
    0
)


empty_synthetic_texts = (
    synthetic_classifier_df[
        "tweet"
    ]
    .str.len()
    .eq(0)
    .sum()
)


assert (
    empty_synthetic_texts
    ==
    0
), (
    f"Found {empty_synthetic_texts} empty synthetic tweets."
)


# ============================================================
# EXPECTED SYNTHETIC CLASS DISTRIBUTION
# ============================================================

EXPECTED_SYNTHETIC_COUNTS = {
    2: 500,
    3: 537,
    4: 411,
}


actual_synthetic_counts = (
    synthetic_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    len(
        synthetic_classifier_df
    )
    ==
    SYNTHETIC_1X_SIZE
)


assert (
    len(
        synthetic_classifier_df
    )
    ==
    1448
)


assert (
    actual_synthetic_counts
    ==
    EXPECTED_SYNTHETIC_COUNTS
), (
    f"Unexpected synthetic class counts.\n"
    f"Expected: {EXPECTED_SYNTHETIC_COUNTS}\n"
    f"Actual:   {actual_synthetic_counts}"
)


assert (
    set(
        synthetic_classifier_df[
            "label"
        ]
        .unique()
    )
    ==
    {
        2,
        3,
        4,
    }
)


print(
    "\nSynthetic classifier samples:",
    len(
        synthetic_classifier_df
    )
)


print(
    "\nSynthetic class distribution:"
)


display(
    synthetic_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nFinal frozen QC-filtered synthetic set: VERIFIED"
)


# ============================================================
# VERIFY ORIGINAL TRAINING DATA
# ============================================================

assert (
    len(
        train_df
    )
    ==
    ORIGINAL_TRAIN_SIZE
)


assert (
    len(
        train_df
    )
    ==
    6408
)


EXPECTED_ORIGINAL_COUNTS = {
    0: 1537,
    1: 3423,
    2: 500,
    3: 537,
    4: 411,
}


actual_original_counts = (
    train_df[
        "label"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_original_counts
    ==
    EXPECTED_ORIGINAL_COUNTS
)


# ============================================================
# BUILD FINAL 1x QC-FILTERED AUGMENTED DATAFRAME
#
# IMPORTANT:
# Constructed ONCE.
# Do not rebuild or resample inside the seed loop.
# ============================================================

train_aug_df = pd.concat(
    [
        train_df[
            [
                "tweet",
                "label",
            ]
        ]
        .copy(),

        synthetic_classifier_df[
            [
                "tweet",
                "label",
            ]
        ]
        .copy(),
    ],

    ignore_index=True,
)


# ============================================================
# NORMALIZE DATATYPES
# ============================================================

train_aug_df[
    "tweet"
] = (
    train_aug_df[
        "tweet"
    ]
    .astype(str)
)


train_aug_df[
    "label"
] = (
    train_aug_df[
        "label"
    ]
    .astype(int)
)


# ============================================================
# VERIFY FINAL TRAINING SIZE
# ============================================================

assert (
    len(
        train_aug_df
    )
    ==
    FINAL_FILTERED_TRAIN_SIZE
)


assert (
    len(
        train_aug_df
    )
    ==
    7856
)


# ============================================================
# EXPECTED FINAL AUGMENTED DISTRIBUTION
# ============================================================

EXPECTED_FILTERED_AUGMENTED_COUNTS = {
    0: 1537,
    1: 3423,
    2: 1000,
    3: 1074,
    4: 822,
}


actual_augmented_counts = (
    train_aug_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_augmented_counts
    ==
    EXPECTED_FILTERED_AUGMENTED_COUNTS
), (
    f"Unexpected final augmented distribution.\n"
    f"Expected: {EXPECTED_FILTERED_AUGMENTED_COUNTS}\n"
    f"Actual:   {actual_augmented_counts}"
)


# ============================================================
# VERIFY NO MISSING VALUES
# ============================================================

assert (
    train_aug_df[
        "tweet"
    ]
    .isna()
    .sum()
    ==
    0
)


assert (
    train_aug_df[
        "label"
    ]
    .isna()
    .sum()
    ==
    0
)


# ============================================================
# CREATE DETERMINISTIC DATAFRAME HASH
#
# This proves that all three seeds use exactly the same
# augmented training dataframe.
# ============================================================

def dataframe_sha256(df):

    canonical_text = (
        df[
            [
                "tweet",
                "label",
            ]
        ]
        .astype(
            {
                "tweet":
                    str,

                "label":
                    int,
            }
        )
        .to_csv(
            index=False,
            lineterminator="\n",
        )
    )


    return hashlib.sha256(
        canonical_text.encode(
            "utf-8"
        )
    ).hexdigest()


FILTERED_TRAINING_DATA_HASH = (
    dataframe_sha256(
        train_aug_df
    )
)


# ============================================================
# DISPLAY FINAL DATAFRAME VERIFICATION
# ============================================================

print("\n" + "=" * 80)

print(
    "FINAL QC-FILTERED 1x AUGMENTED TRAINING DATA"
)

print("=" * 80)


print(
    "Original RUHSOLD samples:",
    len(
        train_df
    )
)


print(
    "QC-filtered synthetic samples:",
    len(
        synthetic_classifier_df
    )
)


print(
    "Final augmented samples:",
    len(
        train_aug_df
    )
)


print(
    "\nOriginal class distribution:"
)


display(
    train_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nSynthetic class distribution:"
)


display(
    synthetic_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nFinal augmented class distribution:"
)


display(
    train_aug_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nFixed training-data SHA256:"
)

print(
    FILTERED_TRAINING_DATA_HASH
)


print(
    "\nThe same fixed dataframe will be used "
    "for seeds 42, 43, and 44."
)


# ============================================================
# CREATE DATASET OBJECTS
#
# IMPORTANT:
# Test set is intentionally NOT instantiated here.
# It is reserved for final evaluation only.
# ============================================================

train_aug_dataset = (
    RUHSOLDDataset(
        dataframe=(
            train_aug_df
        ),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
)


val_dataset = (
    RUHSOLDDataset(
        dataframe=val_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
)


# ============================================================
# VERIFY DATASET OBJECT SIZES
# ============================================================

assert (
    len(
        train_aug_dataset
    )
    ==
    FINAL_FILTERED_TRAIN_SIZE
)


assert (
    len(
        train_aug_dataset
    )
    ==
    7856
)


assert (
    len(
        val_dataset
    )
    ==
    801
)


# ============================================================
# VERIFY TRAINING LABELS INSIDE DATASET OBJECT
# ============================================================

dataset_train_counts = {

    int(label):
        int(count)

    for label, count

    in zip(

        *np.unique(

            np.asarray(
                train_aug_dataset.labels,
                dtype=int,
            ),

            return_counts=True,
        )
    )
}


assert (
    dataset_train_counts
    ==
    EXPECTED_FILTERED_AUGMENTED_COUNTS
)


# ============================================================
# VERIFY VALIDATION DISTRIBUTION
# ============================================================

EXPECTED_VAL_COUNTS = {
    0: 192,
    1: 428,
    2: 63,
    3: 67,
    4: 51,
}


dataset_val_counts = {

    int(label):
        int(count)

    for label, count

    in zip(

        *np.unique(

            np.asarray(
                val_dataset.labels,
                dtype=int,
            ),

            return_counts=True,
        )
    )
}


assert (
    dataset_val_counts
    ==
    EXPECTED_VAL_COUNTS
)


# ============================================================
# VERIFY HASH AFTER DATASET CREATION
# ============================================================

current_filtered_hash = (
    dataframe_sha256(
        train_aug_df
    )
)


assert (
    current_filtered_hash
    ==
    FILTERED_TRAINING_DATA_HASH
)


# ============================================================
# FINAL DATA AUDIT
# ============================================================

print("\n" + "=" * 80)

print(
    "QC-FILTERED DATASET OBJECT VERIFICATION"
)

print("=" * 80)


print(
    "Augmented training dataset:",
    len(
        train_aug_dataset
    )
)


print(
    "Validation dataset:",
    len(
        val_dataset
    )
)


print(
    "\nTraining distribution:"
)

print(
    dataset_train_counts
)


print(
    "Validation distribution:"
)

print(
    dataset_val_counts
)


print(
    "\nTraining-data SHA256:"
)

print(
    current_filtered_hash
)


print(
    "\nTEST SET STATUS:"
)

print(
    "Not instantiated. Reserved for final evaluation only."
)


print(
    "\nFINAL QC-FILTERED 1x TRAINING DATA VERIFIED."
)

QC-FILTERED 1x SYNTHETIC DATA VERIFICATION
Frozen synthetic file:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/final_1x_synthetic_training_set.csv

Raw synthetic dataset shape: (1448, 42)
Available columns:
['production_candidate_id', 'batch_number', 'sample_index', 'class_id', 'target_label', 'generation_seed', 'demonstration_seed', 'prompt_version', 'prompting_strategy', 'demonstration_source', 'demonstration_pool_size', 'demo_1', 'demo_2', 'demo_3', 'demo_4', 'generated_text', 'source_batch_file', 'word_count', 'is_empty', 'has_urdu_script', 'has_latin_content', 'is_non_linguistic', 'is_too_short', 'is_too_long', 'is_exact_duplicate', 'stage0_rejection_reason', 'stage0_pass', 'xlmr_predicted_class_id', 'xlmr_predicted_class', 'stage1_pass', 'stage1_rejection_reason', 'stage2_max_similarity', 'stage2_mean_top10_similarity', 'stage2_nearest_real_text', 'stage2_nearest_real_similarity', 'stage2_pass', 'stage2_rejection_reason', 's

,label,count
0,2,500
1,3,537
2,4,411



Final frozen QC-filtered synthetic set: VERIFIED

FINAL QC-FILTERED 1x AUGMENTED TRAINING DATA
Original RUHSOLD samples: 6408
QC-filtered synthetic samples: 1448
Final augmented samples: 7856

Original class distribution:


,label,count
0,0,1537
1,1,3423
2,2,500
3,3,537
4,4,411



Synthetic class distribution:


,label,count
0,2,500
1,3,537
2,4,411



Final augmented class distribution:


,label,count
0,0,1537
1,1,3423
2,2,1000
3,3,1074
4,4,822



Fixed training-data SHA256:
41eb49f4708aeacf53ed6d4caff4bfd0f69762c193fb4b96588d9d632e227f06

The same fixed dataframe will be used for seeds 42, 43, and 44.

QC-FILTERED DATASET OBJECT VERIFICATION
Augmented training dataset: 7856
Validation dataset: 801

Training distribution:
{0: 1537, 1: 3423, 2: 1000, 3: 1074, 4: 822}
Validation distribution:
{0: 192, 1: 428, 2: 63, 3: 67, 4: 51}

Training-data SHA256:
41eb49f4708aeacf53ed6d4caff4bfd0f69762c193fb4b96588d9d632e227f06

TEST SET STATUS:
Not instantiated. Reserved for final evaluation only.

FINAL QC-FILTERED 1x TRAINING DATA VERIFIED.


In [12]:
# ============================================================
# CORRECTED QC-FILTERED 1x AUGMENTED XLM-R
# THREE-SEED VALIDATION + PER-CLASS EXPERIMENT
#
# CORRECTIONS:
# 1. XLM-R pretrained weights loaded normally in FP32.
# 2. BF16 used ONLY as Trainer mixed precision.
# 3. Same fixed QC-filtered training dataframe reused
#    across seeds 42, 43, and 44.
# 4. Training-data SHA256 verified before every seed.
# 5. Warmup ratio preserved using 185 optimizer steps.
# 6. Existing completed checkpoints are protected.
# 7. Test set remains untouched.
# ============================================================

import gc
import math
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    classification_report,
)

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)


# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

SEEDS = [
    42,
    43,
    44,
]


# ============================================================
# CORRECTED OUTPUT DIRECTORIES
#
# IMPORTANT:
# Use a NEW directory.
# Do not overwrite the old BF16-loaded experiment.
# ============================================================

FILTERED_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "classifier"
    / "outputs"
    / "xlm_roberta_filtered_1x_corrected"
)


FILTERED_RESULTS_ROOT = (
    PROJECT_ROOT
    / "classifier"
    / "outputs"
    / "xlm_roberta_filtered_1x_corrected_results"
)


FILTERED_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FILTERED_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LABEL CONFIGURATION
# ============================================================

METRIC_LABELS = [
    0,
    1,
    2,
    3,
    4,
]


TARGET_NAMES = [
    "Abusive/Offensive",
    "Normal",
    "Religious Hate",
    "Sexism",
    "Profane",
]


CHECKPOINT_SELECTION_METRIC = (
    "macro_f1"
)


# ============================================================
# CANONICAL MATCHED BATCH CONFIGURATION
#
# Same effective batch size as:
# - Original XLM-R baseline
# - Corrected unfiltered XLM-R
# ============================================================

TRAIN_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 1

EVAL_BATCH_SIZE = 16


EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    *
    GRADIENT_ACCUMULATION_STEPS
)


assert (
    EFFECTIVE_BATCH_SIZE
    ==
    16
)


# ============================================================
# FINAL TRAINING SIZE
# ============================================================

FINAL_FILTERED_TRAIN_SIZE = 7856


assert (
    len(train_aug_df)
    ==
    FINAL_FILTERED_TRAIN_SIZE
)


assert (
    len(train_aug_dataset)
    ==
    FINAL_FILTERED_TRAIN_SIZE
)


assert (
    len(val_dataset)
    ==
    801
)


# ============================================================
# WARMUP CALCULATION
#
# 7856 / 16 = 491 optimizer steps per epoch
# 491 × 10 = 4910 maximum optimizer steps
#
# Same frozen Optuna warmup ratio:
# 0.03756172525242821
#
# ceil(4910 × ratio) = 185
# ============================================================

OPTIMIZER_STEPS_PER_EPOCH = math.ceil(
    FINAL_FILTERED_TRAIN_SIZE
    /
    EFFECTIVE_BATCH_SIZE
)


MAX_OPTIMIZER_STEPS = (
    OPTIMIZER_STEPS_PER_EPOCH
    *
    FINAL_MAX_EPOCHS
)


FILTERED_WARMUP_STEPS = math.ceil(
    MAX_OPTIMIZER_STEPS
    *
    BEST_WARMUP_RATIO
)


assert (
    OPTIMIZER_STEPS_PER_EPOCH
    ==
    491
)


assert (
    MAX_OPTIMIZER_STEPS
    ==
    4910
)


assert (
    FILTERED_WARMUP_STEPS
    ==
    185
)


# ============================================================
# FIXED DATASET HASH
#
# FILTERED_TRAINING_DATA_HASH was created in the previous
# data-preparation block.
# ============================================================

assert (
    "FILTERED_TRAINING_DATA_HASH"
    in globals()
), (
    "FILTERED_TRAINING_DATA_HASH is missing. "
    "Run the corrected filtered data-preparation block first."
)


CURRENT_FILTERED_HASH = (
    dataframe_sha256(
        train_aug_df
    )
)


assert (
    CURRENT_FILTERED_HASH
    ==
    FILTERED_TRAINING_DATA_HASH
), (
    "Filtered training dataframe changed before training."
)


# ============================================================
# PRE-TRAINING AUDIT
# ============================================================

print("=" * 80)

print(
    "CORRECTED QC-FILTERED 1x XLM-R "
    "- PRE-TRAINING AUDIT"
)

print("=" * 80)


print(
    "Training samples:",
    len(train_aug_dataset)
)


print(
    "Validation samples:",
    len(val_dataset)
)


print(
    "Seeds:",
    SEEDS
)


print(
    "\nFrozen hyperparameters:"
)


print(
    "Learning rate:",
    BEST_LEARNING_RATE
)


print(
    "Weight decay:",
    BEST_WEIGHT_DECAY
)


print(
    "Warmup ratio:",
    BEST_WARMUP_RATIO
)


print(
    "Warmup steps:",
    FILTERED_WARMUP_STEPS
)


print(
    "\nCanonical batch configuration:"
)


print(
    "Physical train batch:",
    TRAIN_BATCH_SIZE
)


print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)


print(
    "Effective batch:",
    EFFECTIVE_BATCH_SIZE
)


print(
    "Evaluation batch:",
    EVAL_BATCH_SIZE
)


print(
    "\nOptimizer steps per epoch:",
    OPTIMIZER_STEPS_PER_EPOCH
)


print(
    "Maximum optimizer steps:",
    MAX_OPTIMIZER_STEPS
)


print(
    "\nFixed training-data hash:"
)


print(
    FILTERED_TRAINING_DATA_HASH
)


print(
    "\nPre-training configuration audit: PASSED"
)


# ============================================================
# RESULT STORAGE
# ============================================================

overall_results = []

per_class_results = []


# ============================================================
# CHECK FOR EXISTING CORRECTED SEEDS
#
# We do NOT automatically delete existing completed runs.
# ============================================================

existing_seed_checkpoints = {}


for seed in SEEDS:

    seed_dir = (
        FILTERED_OUTPUT_ROOT
        /
        f"seed_{seed}"
    )

    checkpoints = (
        sorted(
            seed_dir.glob(
                "checkpoint-*"
            )
        )
        if seed_dir.exists()
        else []
    )

    if len(checkpoints) > 0:

        existing_seed_checkpoints[
            seed
        ] = checkpoints


print("\n" + "=" * 80)

print(
    "EXISTING CORRECTED CHECKPOINT AUDIT"
)

print("=" * 80)


if not existing_seed_checkpoints:

    print(
        "No existing corrected filtered checkpoints."
    )

else:

    for seed, checkpoints in (
        existing_seed_checkpoints.items()
    ):

        print(
            f"\nSeed {seed}:"
        )

        for checkpoint in checkpoints:

            print(
                checkpoint
            )


# ============================================================
# SAFETY
#
# If corrected checkpoints already exist, STOP rather than
# silently deleting them.
#
# If this is the first corrected run, execution continues.
# ============================================================

if existing_seed_checkpoints:

    raise RuntimeError(
        "\nExisting corrected filtered checkpoint(s) detected.\n"
        "Nothing has been deleted.\n\n"
        "Do not rerun completed seeds automatically.\n"
        "Inspect the checkpoint audit above first."
    )


# ============================================================
# THREE-SEED TRAINING
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 80)

    print(
        f"CORRECTED QC-FILTERED 1x XLM-R "
        f"- SEED {seed}"
    )

    print("=" * 80)


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    set_seed(
        seed
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # ========================================================
    # VERIFY FIXED TRAINING DATA BEFORE THIS SEED
    #
    # Critical Issue 4 check:
    # every seed must receive exactly the same dataframe.
    # ========================================================

    seed_training_hash = (
        dataframe_sha256(
            train_aug_df
        )
    )


    print(
        "Training-data hash:",
        seed_training_hash
    )


    assert (
        seed_training_hash
        ==
        FILTERED_TRAINING_DATA_HASH
    ), (
        f"Training dataframe changed before seed {seed}."
    )


    print(
        "Fixed dataset integrity: PASSED"
    )


    # ========================================================
    # SEED OUTPUT DIRECTORY
    # ========================================================

    seed_output_dir = (
        FILTERED_OUTPUT_ROOT
        /
        f"seed_{seed}"
    )


    # Because the pre-training audit confirmed that this
    # corrected seed did not previously exist, creating a
    # fresh directory is safe.

    if seed_output_dir.exists():

        print(
            "Removing empty/incomplete corrected seed directory:"
        )

        print(
            seed_output_dir
        )

        shutil.rmtree(
            seed_output_dir
        )


    seed_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # LOAD FRESH XLM-R IN NORMAL PRETRAINED PRECISION
    #
    # CRITICAL ISSUE 1 FIX:
    #
    # DO NOT use:
    #
    # dtype=torch.bfloat16
    #
    # here.
    #
    # Pretrained model weights are loaded normally.
    # BF16 is enabled only through TrainingArguments.
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(

            MODEL_NAME,

            num_labels=NUM_LABELS,

            id2label=id2label,

            label2id=label2id,
        )
    )


    # ========================================================
    # VERIFY PRE-TRAINER MODEL PRECISION
    # ========================================================

    model_parameter_dtype = (
        next(
            model.parameters()
        ).dtype
    )


    print(
        "\nModel parameter dtype before Trainer:",
        model_parameter_dtype
    )


    assert (
        model_parameter_dtype
        ==
        torch.float32
    ), (
        f"Expected FP32 pretrained model loading, "
        f"but found {model_parameter_dtype}"
    )


    print(
        "FP32 pretrained model loading: VERIFIED"
    )


    # ========================================================
    # TRAINING ARGUMENTS
    # ========================================================

    training_args = TrainingArguments(

        output_dir=str(
            seed_output_dir
        ),

        # ----------------------------------------------------
        # Evaluation / logging
        # ----------------------------------------------------

        eval_strategy="epoch",

        logging_strategy="epoch",

        # ----------------------------------------------------
        # Keep validation-selected best checkpoint
        # ----------------------------------------------------

        save_strategy="best",

        save_total_limit=1,

        save_only_model=True,

        # ----------------------------------------------------
        # Frozen Optuna hyperparameters
        # ----------------------------------------------------

        learning_rate=(
            BEST_LEARNING_RATE
        ),

        weight_decay=(
            BEST_WEIGHT_DECAY
        ),

        # ----------------------------------------------------
        # Canonical matched batch configuration
        # ----------------------------------------------------

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        # ----------------------------------------------------
        # Same frozen warmup ratio converted for 7856 samples
        # ----------------------------------------------------

        warmup_steps=(
            FILTERED_WARMUP_STEPS
        ),

        # ----------------------------------------------------
        # Maximum training duration
        # ----------------------------------------------------

        num_train_epochs=(
            FINAL_MAX_EPOCHS
        ),

        # ----------------------------------------------------
        # Validation-only checkpoint selection
        # ----------------------------------------------------

        load_best_model_at_end=True,

        metric_for_best_model=(
            CHECKPOINT_SELECTION_METRIC
        ),

        greater_is_better=True,

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------

        seed=seed,

        data_seed=seed,

        # ----------------------------------------------------
        # MIXED PRECISION
        #
        # Model weights were loaded normally in FP32.
        # BF16 is used ONLY here during training.
        # ----------------------------------------------------

        bf16=(
            torch.cuda.is_available()
        ),

        fp16=False,

        # ----------------------------------------------------
        # Misc
        # ----------------------------------------------------

        report_to="none",

        disable_tqdm=False,
    )


    # ========================================================
    # TRAINING ARGUMENT AUDIT
    # ========================================================

    print(
        "\nTraining configuration:"
    )


    print(
        "Physical train batch:",
        training_args.per_device_train_batch_size
    )


    print(
        "Gradient accumulation:",
        training_args.gradient_accumulation_steps
    )


    print(
        "Effective batch:",
        (
            training_args.per_device_train_batch_size
            *
            training_args.gradient_accumulation_steps
        )
    )


    print(
        "Evaluation batch:",
        training_args.per_device_eval_batch_size
    )


    print(
        "Warmup steps:",
        training_args.warmup_steps
    )


    print(
        "Trainer BF16:",
        training_args.bf16
    )


    assert (
        training_args.per_device_train_batch_size
        *
        training_args.gradient_accumulation_steps
        ==
        16
    )


    assert (
        training_args.warmup_steps
        ==
        185
    )


    # ========================================================
    # TRAINER
    # ========================================================

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=(
            train_aug_dataset
        ),

        eval_dataset=(
            val_dataset
        ),

        data_collator=(
            data_collator
        ),

        compute_metrics=(
            compute_metrics
        ),

        processing_class=(
            tokenizer
        ),

        callbacks=[

            EarlyStoppingCallback(

                early_stopping_patience=(
                    FINAL_EARLY_STOPPING_PATIENCE
                ),

                early_stopping_threshold=0.0,
            )
        ],
    )


    # ========================================================
    # TRAIN
    # ========================================================

    print(
        "\nStarting corrected filtered training..."
    )


    start_time = time.time()


    trainer.train()


    training_time = (
        time.time()
        -
        start_time
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    best_checkpoint = (
        trainer
        .state
        .best_model_checkpoint
    )


    best_validation_macro_f1 = (
        trainer
        .state
        .best_metric
    )


    epoch_reached = (
        trainer
        .state
        .epoch
    )


    if best_checkpoint is None:

        raise RuntimeError(
            f"No best checkpoint recorded "
            f"for seed {seed}."
        )


    best_checkpoint_path = (
        Path(
            best_checkpoint
        )
    )


    assert (
        best_checkpoint_path.exists()
    )


    print(
        "\nBest checkpoint:"
    )

    print(
        best_checkpoint_path
    )


    print(
        "Best validation Macro F1:",
        f"{best_validation_macro_f1:.4f}"
    )


    print(
        "Epoch reached:",
        epoch_reached
    )


    print(
        "Training time:",
        f"{training_time / 60:.2f} minutes"
    )


    # ========================================================
    # VALIDATION PREDICTIONS USING BEST MODEL
    # ========================================================

    prediction_output = (
        trainer.predict(
            val_dataset
        )
    )


    y_true = (
        prediction_output.label_ids
    )


    y_pred = np.argmax(
        prediction_output.predictions,
        axis=1,
    )


    assert (
        len(y_true)
        ==
        801
    )


    assert (
        len(y_pred)
        ==
        801
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(

        y_true,

        y_pred,

        labels=(
            METRIC_LABELS
        ),

        target_names=[
            id2label[
                class_id
            ]
            for class_id
            in METRIC_LABELS
        ],

        output_dict=True,

        zero_division=0,
    )


    report_df = (
        pd.DataFrame(
            report
        )
        .transpose()
    )


    print(
        f"\nPER-CLASS VALIDATION RESULTS "
        f"- SEED {seed}"
    )


    display(
        report_df.round(4)
    )


    # ========================================================
    # STORE OVERALL RESULTS
    # ========================================================

    overall_result = {

        "seed":
            seed,

        "training_data_hash":
            seed_training_hash,

        "model_loading_dtype":
            str(
                model_parameter_dtype
            ),

        "trainer_bf16":
            training_args.bf16,

        "best_checkpoint":
            str(
                best_checkpoint_path
            ),

        "epoch_reached":
            epoch_reached,

        "best_validation_macro_f1":
            best_validation_macro_f1,

        "validation_accuracy":
            report[
                "accuracy"
            ],

        "validation_macro_precision":
            report[
                "macro avg"
            ][
                "precision"
            ],

        "validation_macro_recall":
            report[
                "macro avg"
            ][
                "recall"
            ],

        "validation_macro_f1":
            report[
                "macro avg"
            ][
                "f1-score"
            ],

        "validation_weighted_precision":
            report[
                "weighted avg"
            ][
                "precision"
            ],

        "validation_weighted_recall":
            report[
                "weighted avg"
            ][
                "recall"
            ],

        "validation_weighted_f1":
            report[
                "weighted avg"
            ][
                "f1-score"
            ],

        "training_time_minutes":
            training_time / 60,
    }


    overall_results.append(
        overall_result
    )


    # ========================================================
    # STORE PER-CLASS RESULTS
    # ========================================================

    seed_per_class_rows = []


    for class_id in METRIC_LABELS:

        class_name = (
            id2label[
                class_id
            ]
        )


        class_metrics = (
            report[
                class_name
            ]
        )


        row = {

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        }


        seed_per_class_rows.append(
            row
        )


        per_class_results.append(
            row
        )


    # ========================================================
    # SAVE VALIDATION PREDICTIONS
    # ========================================================

    prediction_df = pd.DataFrame({

        "true_label_id":
            y_true,

        "predicted_label_id":
            y_pred,

        "true_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_true
            ],

        "predicted_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_pred
            ],
    })


    prediction_df.to_csv(

        FILTERED_RESULTS_ROOT
        /
        (
            f"seed_{seed}_"
            "validation_predictions.csv"
        ),

        index=False,
    )


    # ========================================================
    # SAVE CUMULATIVE RESULTS AFTER EACH SEED
    # ========================================================

    pd.DataFrame(
        overall_results
    ).to_csv(

        FILTERED_RESULTS_ROOT
        /
        "filtered_1x_3seed_overall_results.csv",

        index=False,
    )


    pd.DataFrame(
        per_class_results
    ).to_csv(

        FILTERED_RESULTS_ROOT
        /
        "filtered_1x_3seed_per_class_results.csv",

        index=False,
    )


    # ========================================================
    # CHECKPOINT RETENTION
    # ========================================================

    print(
        "\nCheckpoint safely retained:"
    )


    print(
        best_checkpoint_path
    )


    assert (
        best_checkpoint_path.exists()
    )


    # ========================================================
    # MEMORY CLEANUP ONLY
    #
    # DO NOT DELETE CHECKPOINT.
    # ========================================================

    del prediction_output

    del trainer

    del model


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    assert (
        best_checkpoint_path.exists()
    )


    print(
        "Checkpoint confirmed after memory cleanup:"
    )


    print(
        best_checkpoint_path
    )


    # ========================================================
    # DISK REPORT
    # ========================================================

    total, used, free = (
        shutil.disk_usage(
            "/home/jovyan"
        )
    )


    print(
        "Disk free:",
        f"{free / 1024**3:.2f} GB"
    )


# ============================================================
# THREE-SEED SUMMARY
# ============================================================

overall_results_df = (
    pd.DataFrame(
        overall_results
    )
)


assert (
    len(
        overall_results_df
    )
    ==
    3
)


print("\n" + "=" * 80)

print(
    "CORRECTED QC-FILTERED 1x XLM-R "
    "- THREE-SEED EXPERIMENT COMPLETE"
)

print("=" * 80)


print(
    "\nPer-seed overall results:"
)


display(
    overall_results_df.round(4)
)


# ============================================================
# VERIFY IDENTICAL DATA HASH ACROSS ALL SEEDS
# ============================================================

unique_training_hashes = (
    overall_results_df[
        "training_data_hash"
    ]
    .nunique()
)


assert (
    unique_training_hashes
    ==
    1
), (
    "Different training datasets were used "
    "across seeds."
)


assert (
    overall_results_df[
        "training_data_hash"
    ]
    .iloc[0]
    ==
    FILTERED_TRAINING_DATA_HASH
)


print(
    "\nSame fixed training dataset across "
    "all three seeds: VERIFIED"
)


# ============================================================
# VERIFY MODEL LOADING PRECISION ACROSS SEEDS
# ============================================================

assert (
    overall_results_df[
        "model_loading_dtype"
    ]
    .eq(
        "torch.float32"
    )
    .all()
)


assert (
    overall_results_df[
        "trainer_bf16"
    ]
    .all()
)


print(
    "FP32 pretrained loading across all seeds: VERIFIED"
)


print(
    "BF16 Trainer mixed precision across all seeds: VERIFIED"
)


# ============================================================
# OVERALL MEAN ± STANDARD DEVIATION
# ============================================================

summary_columns = [

    "validation_accuracy",

    "validation_macro_precision",

    "validation_macro_recall",

    "validation_macro_f1",

    "validation_weighted_f1",
]


overall_summary_df = (

    overall_results_df[
        summary_columns
    ]

    .agg(
        [
            "mean",
            "std",
        ]
    )
)


print(
    "\nThree-seed overall validation summary:"
)


display(
    overall_summary_df.round(4)
)


print(
    "\nMean validation Macro F1:",
    f'{overall_results_df["validation_macro_f1"].mean():.4f}'
)


print(
    "Validation Macro F1 standard deviation:",
    f'{overall_results_df["validation_macro_f1"].std(ddof=1):.4f}'
)


# ============================================================
# PER-CLASS SUMMARY
# ============================================================

per_class_results_df = (
    pd.DataFrame(
        per_class_results
    )
)


assert (
    len(
        per_class_results_df
    )
    ==
    15
)


per_class_summary_df = (

    per_class_results_df

    .groupby(
        [
            "class_id",
            "class_name",
        ]
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),
    )

    .reset_index()
)


print(
    "\nThree-seed per-class validation summary:"
)


display(
    per_class_summary_df.round(4)
)


# ============================================================
# SAVE FINAL SUMMARIES
# ============================================================

overall_summary_df.to_csv(

    FILTERED_RESULTS_ROOT
    /
    "filtered_1x_3seed_overall_summary.csv"
)


per_class_summary_df.to_csv(

    FILTERED_RESULTS_ROOT
    /
    "filtered_1x_3seed_per_class_summary.csv",

    index=False,
)


# ============================================================
# FINAL CHECKPOINT VERIFICATION
# ============================================================

print("\n" + "=" * 80)

print(
    "RETAINED CORRECTED QC-FILTERED CHECKPOINTS"
)

print("=" * 80)


for seed in SEEDS:

    seed_dir = (
        FILTERED_OUTPUT_ROOT
        /
        f"seed_{seed}"
    )


    checkpoints = (
        sorted(
            seed_dir.glob(
                "checkpoint-*"
            )
        )
    )


    print(
        f"\nSeed {seed}:"
    )


    for checkpoint in checkpoints:

        print(
            checkpoint
        )


    assert (
        len(checkpoints)
        ==
        1
    ), (
        f"Expected exactly one retained checkpoint "
        f"for seed {seed}, found {len(checkpoints)}."
    )


print("\n" + "=" * 80)

print(
    "FINAL CORRECTED FILTERED EXPERIMENT VERIFIED"
)

print("=" * 80)


print(
    "FP32 pretrained loading: VERIFIED"
)


print(
    "BF16 Trainer mixed precision: VERIFIED"
)


print(
    "Warmup steps: 185"
)


print(
    "Effective batch size: 16"
)


print(
    "Same fixed training data across seeds: VERIFIED"
)


print(
    "Test set used during training: NO"
)


print(
    "\nResults saved to:"
)


print(
    FILTERED_RESULTS_ROOT
)

CORRECTED QC-FILTERED 1x XLM-R - PRE-TRAINING AUDIT
Training samples: 7856
Validation samples: 801
Seeds: [42, 43, 44]

Frozen hyperparameters:
Learning rate: 2.881129057462248e-05
Weight decay: 0.03348739395854274
Warmup ratio: 0.03756172525242821
Warmup steps: 185

Canonical batch configuration:
Physical train batch: 16
Gradient accumulation: 1
Effective batch: 16
Evaluation batch: 16

Optimizer steps per epoch: 491
Maximum optimizer steps: 4910

Fixed training-data hash:
41eb49f4708aeacf53ed6d4caff4bfd0f69762c193fb4b96588d9d632e227f06

Pre-training configuration audit: PASSED

EXISTING CORRECTED CHECKPOINT AUDIT
No existing corrected filtered checkpoints.

CORRECTED QC-FILTERED 1x XLM-R - SEED 42
Training-data hash: 41eb49f4708aeacf53ed6d4caff4bfd0f69762c193fb4b96588d9d632e227f06
Fixed dataset integrity: PASSED


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[HAMI-core Msg(1624:140454015981376:multiprocess_memory_limit.c:455)]: Calling exit


Model parameter dtype before Trainer: torch.float32
FP32 pretrained model loading: VERIFIED

Training configuration:
Physical train batch: 16
Gradient accumulation: 1
Effective batch: 16
Evaluation batch: 16
Warmup steps: 185
Trainer BF16: True

Starting corrected filtered training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,1.112536,0.695898,0.759051,0.684024,0.712452,0.694167,0.761490,0.759051,0.758017
2,0.598821,0.646205,0.782772,0.700475,0.726309,0.705906,0.787356,0.782772,0.778463
3,0.496861,0.658737,0.791511,0.701729,0.725195,0.705742,0.795750,0.791511,0.787426
4,0.395522,0.784071,0.754057,0.689900,0.714628,0.692490,0.783834,0.754057,0.761269


...


Same fixed training dataset across all three seeds: VERIFIED
FP32 pretrained loading across all seeds: VERIFIED
BF16 Trainer mixed precision across all seeds: VERIFIED

Three-seed overall validation summary:


,validation_accuracy,validation_macro_precision,validation_macro_recall,validation_macro_f1,validation_weighted_f1
mean,0.7965,0.7151,0.7457,0.7257,0.7956
std,0.0119,0.0133,0.0173,0.0171,0.0149



Mean validation Macro F1: 0.7257
Validation Macro F1 standard deviation: 0.0171

Three-seed per-class validation summary:


,class_id,class_name,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,0,Abusive/Offensive,0.7580,0.0279,0.6319,0.0773,0.6864,0.0369
1,1,Normal,0.8860,0.0254,0.8949,0.0168,0.8901,0.0048
2,2,Religious Hate,0.6376,0.0449,0.7196,0.0092,0.6755,0.0256
3,3,Sexism,0.6847,0.0196,0.7960,0.0565,0.7351,0.0208
4,4,Profane,0.6091,0.0750,0.6863,0.0519,0.6413,0.0171



RETAINED CORRECTED QC-FILTERED CHECKPOINTS

Seed 42:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_filtered_1x_corrected/seed_42/checkpoint-982

Seed 43:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_filtered_1x_corrected/seed_43/checkpoint-4419

Seed 44:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_filtered_1x_corrected/seed_44/checkpoint-3437

FINAL CORRECTED FILTERED EXPERIMENT VERIFIED
FP32 pretrained loading: VERIFIED
BF16 Trainer mixed precision: VERIFIED
Warmup steps: 185
Effective batch size: 16
Same fixed training data across seeds: VERIFIED
Test set used during training: NO

Results saved to:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_filtered_1x_corrected_results


In [40]:
# ============================================================
# QC-FILTERED 1x AUGMENTED XLM-R
# FINAL RUHSOLD TEST-SET EVALUATION
#
# IMPORTANT:
# - NO TRAINING occurs here.
# - Uses ONLY validation-selected retained checkpoints.
# - Test set is used ONLY for final evaluation.
# - Seeds 42, 43, 44 are evaluated independently.
# - Same evaluation protocol as the unfiltered 1x control.
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

TEST_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_test.tsv"
)


# ============================================================
# 2. EXACT VALIDATION-SELECTED QC CHECKPOINTS
# ============================================================

QC_CHECKPOINTS = {

    42: (
        PROJECT_ROOT
        / "outputs"
        / "xlm_roberta_final_1x_augmented"
        / "seed_42"
        / "checkpoint-3437"
    ),

    43: (
        PROJECT_ROOT
        / "outputs"
        / "xlm_roberta_final_1x_augmented"
        / "seed_43"
        / "checkpoint-4419"
    ),

    44: (
        PROJECT_ROOT
        / "outputs"
        / "xlm_roberta_final_1x_augmented"
        / "seed_44"
        / "checkpoint-1473"
    ),
}


# ============================================================
# 3. FINAL TEST RESULTS DIRECTORY
# ============================================================

FINAL_TEST_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "xlm_roberta_final_1x_augmented_results"
    / "final_test"
)

FINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. EXPERIMENT CONFIGURATION
# ============================================================

MODEL_NAME = "FacebookAI/xlm-roberta-base"

MAX_LENGTH = 128

TEST_BATCH_SIZE = 4

SEEDS = [
    42,
    43,
    44,
]


# ============================================================
# 5. RUHSOLD LABEL MAPPING
# ============================================================

id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

label2id = {
    label: class_id
    for class_id, label
    in id2label.items()
}

LABEL_IDS = [
    0,
    1,
    2,
    3,
    4,
]

LABEL_NAMES = [
    id2label[i]
    for i in LABEL_IDS
]


# ============================================================
# 6. CHECKPOINT INTEGRITY CHECK
# ============================================================

print("=" * 70)
print("QC-FILTERED 1x FINAL TEST CHECKPOINTS")
print("=" * 70)


for seed in SEEDS:

    checkpoint_path = (
        QC_CHECKPOINTS[
            seed
        ]
    )

    print(
        f"Seed {seed}:",
        checkpoint_path.exists(),
        checkpoint_path,
    )

    assert checkpoint_path.exists(), (
        f"Checkpoint missing for seed {seed}: "
        f"{checkpoint_path}"
    )

    weight_files = (
        list(
            checkpoint_path.glob(
                "*.safetensors"
            )
        )
        +
        list(
            checkpoint_path.glob(
                "pytorch_model*.bin"
            )
        )
    )

    assert len(weight_files) > 0, (
        f"No model weights found for seed {seed}"
    )

    assert (
        checkpoint_path
        / "config.json"
    ).exists(), (
        f"config.json missing for seed {seed}"
    )


print(
    "\nAll three retained QC-filtered checkpoints verified."
)


# ============================================================
# 7. LOAD ORIGINAL UNTOUCHED RUHSOLD TEST SET
#
# IMPORTANT:
# RUHSOLD_test.tsv HAS NO HEADER ROW.
#
# Therefore:
# header=None
# names=["tweet", "label"]
# ============================================================

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t",
    header=None,
    names=[
        "tweet",
        "label",
    ],
)


# ============================================================
# 8. TEST-SET INTEGRITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL TEST-SET INTEGRITY CHECK")
print("=" * 70)


print(
    "Test samples:",
    len(test_df)
)

print(
    "Missing tweets:",
    test_df[
        "tweet"
    ]
    .isna()
    .sum()
)

print(
    "Missing labels:",
    test_df[
        "label"
    ]
    .isna()
    .sum()
)


assert len(test_df) == 2003


assert (
    test_df[
        "tweet"
    ]
    .isna()
    .sum()
    == 0
)


assert (
    test_df[
        "label"
    ]
    .isna()
    .sum()
    == 0
)


# ------------------------------------------------------------
# Convert labels to integer
# ------------------------------------------------------------

test_df["label"] = (
    test_df[
        "label"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Verify exact expected test distribution
# ------------------------------------------------------------

EXPECTED_TEST_COUNTS = {
    0: 481,
    1: 1070,
    2: 156,
    3: 168,
    4: 128,
}


actual_test_counts = (
    test_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


print(
    "\nTest class distribution:"
)


display(
    test_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


assert (
    actual_test_counts
    ==
    EXPECTED_TEST_COUNTS
), (
    "Unexpected test class distribution.\n"
    f"Expected: {EXPECTED_TEST_COUNTS}\n"
    f"Actual:   {actual_test_counts}"
)


print(
    "\nTest-set integrity check: PASSED"
)


# ============================================================
# 9. TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


def tokenize_function(
    batch
):

    return tokenizer(

        batch[
            "tweet"
        ],

        truncation=True,

        max_length=MAX_LENGTH,
    )


# ============================================================
# 10. CREATE TEST DATASET
# ============================================================

test_dataset = Dataset.from_pandas(

    test_df[
        [
            "tweet",
            "label",
        ]
    ],

    preserve_index=False,
)


test_dataset = (
    test_dataset.map(
        tokenize_function,
        batched=True,
    )
)


test_dataset = (
    test_dataset.rename_column(
        "label",
        "labels",
    )
)


test_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)


data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)


# ============================================================
# 11. TEST DATALOADER
#
# shuffle=False preserves exact test ordering.
# ============================================================

test_loader = DataLoader(

    test_dataset,

    batch_size=TEST_BATCH_SIZE,

    shuffle=False,

    collate_fn=data_collator,
)


print(
    "\nTest DataLoader samples:",
    len(test_dataset)
)

print(
    "Test inference batch size:",
    TEST_BATCH_SIZE
)

print(
    "\nFinal test results directory:"
)

print(
    FINAL_TEST_DIR
)


# ============================================================
# 12. RESULT STORAGE
# ============================================================

all_seed_results = []

all_per_class_results = []

reference_true_labels = None


# ============================================================
# 13. THREE-SEED FINAL TEST LOOP
# ============================================================

for seed in SEEDS:

    print("\n")
    print("=" * 80)

    print(
        f"QC-FILTERED 1x FINAL TEST - SEED {seed}"
    )

    print("=" * 80)


    checkpoint_path = (
        QC_CHECKPOINTS[
            seed
        ]
    )


    # ========================================================
    # GPU CLEANUP
    # ========================================================

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    device = torch.device(

        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    print(
        "Evaluation device:",
        device
    )


    # ========================================================
    # LOAD VALIDATION-SELECTED CHECKPOINT
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(

            checkpoint_path,

            dtype=(
                torch.bfloat16
                if torch.cuda.is_available()
                else torch.float32
            ),
        )
    )


    model = model.to(
        device
    )


    model.eval()


    print(
        "Loaded checkpoint:"
    )

    print(
        checkpoint_path
    )


    print(
        "Model device:",
        next(
            model.parameters()
        ).device
    )


    print(
        "Model dtype:",
        next(
            model.parameters()
        ).dtype
    )


    # ========================================================
    # DIRECT TEST INFERENCE
    # ========================================================

    predictions = []

    true_labels = []


    with torch.inference_mode():

        for batch in test_loader:

            labels = (
                batch.pop(
                    "labels"
                )
            )


            batch = {

                key:
                    value.to(
                        device
                    )

                for key, value
                in batch.items()
            }


            outputs = (
                model(
                    **batch
                )
            )


            batch_predictions = (

                outputs.logits

                .argmax(
                    dim=-1
                )

                .detach()

                .cpu()

                .numpy()
            )


            predictions.extend(
                batch_predictions
            )


            true_labels.extend(
                labels
                .cpu()
                .numpy()
            )


    predictions = np.asarray(
        predictions,
        dtype=int,
    )


    true_labels = np.asarray(
        true_labels,
        dtype=int,
    )


    # ========================================================
    # 14. SEED-LEVEL TEST INTEGRITY CHECK
    # ========================================================

    assert (
        len(predictions)
        == 2003
    )


    assert (
        len(true_labels)
        == 2003
    )


    expected_true_labels = (
        test_df[
            "label"
        ]
        .astype(int)
        .to_numpy()
    )


    assert np.array_equal(

        true_labels,

        expected_true_labels

    ), (
        f"Test label/order mismatch "
        f"for seed {seed}"
    )


    assert set(
        np.unique(
            predictions
        )
    ).issubset(
        set(
            LABEL_IDS
        )
    )


    # --------------------------------------------------------
    # Cross-seed test-label consistency
    # --------------------------------------------------------

    if reference_true_labels is None:

        reference_true_labels = (
            true_labels.copy()
        )

    else:

        assert np.array_equal(

            reference_true_labels,

            true_labels

        ), (
            "Test labels changed between seeds."
        )


    print(
        f"\nSeed {seed} test integrity check: PASSED"
    )

    print(
        "Test predictions completed:",
        len(predictions)
    )


    # ========================================================
    # 15. OVERALL TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        true_labels,
        predictions,
    )


    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = precision_recall_fscore_support(

        true_labels,

        predictions,

        labels=LABEL_IDS,

        average="macro",

        zero_division=0,
    )


    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _
    ) = precision_recall_fscore_support(

        true_labels,

        predictions,

        labels=LABEL_IDS,

        average="weighted",

        zero_division=0,
    )


    print(
        "\n"
        + "-" * 60
    )

    print(
        f"FINAL TEST RESULTS - SEED {seed}"
    )

    print("-" * 60)


    print(
        f"Accuracy        : {accuracy:.4f}"
    )

    print(
        f"Macro Precision : {macro_precision:.4f}"
    )

    print(
        f"Macro Recall    : {macro_recall:.4f}"
    )

    print(
        f"Macro F1        : {macro_f1:.4f}"
    )

    print(
        f"Weighted F1     : {weighted_f1:.4f}"
    )


    # ========================================================
    # 16. CLASSIFICATION REPORT
    # ========================================================

    report_dict = (
        classification_report(

            true_labels,

            predictions,

            labels=LABEL_IDS,

            target_names=LABEL_NAMES,

            output_dict=True,

            zero_division=0,
        )
    )


    report_df = (
        pd.DataFrame(
            report_dict
        )
        .transpose()
    )


    print(
        f"\nPer-class test results - seed {seed}:"
    )


    display(
        report_df.round(4)
    )


    report_df.to_csv(

        FINAL_TEST_DIR
        / (
            f"seed_{seed}_"
            "classification_report.csv"
        )
    )


    # ========================================================
    # 17. SAVE RAW TEST PREDICTIONS
    # ========================================================

    prediction_df = (
        test_df.copy()
    )


    prediction_df[
        "test_index"
    ] = np.arange(
        len(test_df)
    )


    prediction_df[
        "true_label_id"
    ] = true_labels


    prediction_df[
        "predicted_label_id"
    ] = predictions


    prediction_df[
        "true_class"
    ] = [

        id2label[
            int(label)
        ]

        for label
        in true_labels
    ]


    prediction_df[
        "predicted_class"
    ] = [

        id2label[
            int(label)
        ]

        for label
        in predictions
    ]


    prediction_df[
        "correct"
    ] = (
        true_labels
        ==
        predictions
    )


    assert (
        len(prediction_df)
        == 2003
    )


    assert (
        prediction_df[
            "test_index"
        ]
        .duplicated()
        .sum()
        == 0
    )


    prediction_df.to_csv(

        FINAL_TEST_DIR
        / (
            f"seed_{seed}_"
            "test_predictions.csv"
        ),

        index=False,

        encoding="utf-8",
    )


    # ========================================================
    # 18. CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(

        true_labels,

        predictions,

        labels=LABEL_IDS,
    )


    assert (
        cm.sum()
        == 2003
    )


    cm_df = pd.DataFrame(

        cm,

        index=LABEL_NAMES,

        columns=LABEL_NAMES,
    )


    cm_df.to_csv(

        FINAL_TEST_DIR
        / (
            f"seed_{seed}_"
            "confusion_matrix.csv"
        )
    )


    # ========================================================
    # 19. STORE OVERALL SEED RESULT
    # ========================================================

    seed_result = {

        "seed":
            seed,

        "checkpoint":
            str(
                checkpoint_path
            ),

        "test_samples":
            len(
                true_labels
            ),

        "test_accuracy":
            accuracy,

        "test_macro_precision":
            macro_precision,

        "test_macro_recall":
            macro_recall,

        "test_macro_f1":
            macro_f1,

        "test_weighted_precision":
            weighted_precision,

        "test_weighted_recall":
            weighted_recall,

        "test_weighted_f1":
            weighted_f1,
    }


    all_seed_results.append(
        seed_result
    )


    # ========================================================
    # 20. STORE PER-CLASS RESULTS
    # ========================================================

    for class_id in LABEL_IDS:

        class_name = (
            id2label[
                class_id
            ]
        )


        class_metrics = (
            report_dict[
                class_name
            ]
        )


        all_per_class_results.append({

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        })


    # ========================================================
    # 21. SAVE INCREMENTALLY AFTER EACH SEED
    # ========================================================

    pd.DataFrame(
        all_seed_results
    ).to_csv(

        FINAL_TEST_DIR
        / "three_seed_test_results.csv",

        index=False,
    )


    pd.DataFrame(
        all_per_class_results
    ).to_csv(

        FINAL_TEST_DIR
        / "three_seed_per_class_raw.csv",

        index=False,
    )


    # ========================================================
    # 22. CLEAN GPU BEFORE NEXT SEED
    # ========================================================

    del outputs
    del model


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    print(
        f"\nSeed {seed} final test evaluation safely saved."
    )


# ============================================================
# 23. CROSS-SEED TEST-LABEL CONSISTENCY
# ============================================================

assert np.array_equal(

    reference_true_labels,

    test_df[
        "label"
    ]
    .astype(int)
    .to_numpy()
)


print(
    "\n"
    + "=" * 70
)

print(
    "CROSS-SEED TEST-LABEL CONSISTENCY: PASSED"
)

print("=" * 70)


# ============================================================
# 24. THREE-SEED OVERALL TEST RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_seed_results
)


assert (
    len(results_df)
    == 3
)


assert (
    results_df[
        "seed"
    ]
    .tolist()
    ==
    SEEDS
)


print(
    "\n"
    + "=" * 80
)

print(
    "QC-FILTERED 1x - FINAL THREE-SEED TEST RESULTS"
)

print("=" * 80)


display(
    results_df.round(4)
)


results_df.to_csv(

    FINAL_TEST_DIR
    / "three_seed_test_results.csv",

    index=False,
)


# ============================================================
# 25. THREE-SEED MEAN ± SAMPLE STANDARD DEVIATION
# ============================================================

metric_columns = [

    "test_accuracy",

    "test_macro_precision",

    "test_macro_recall",

    "test_macro_f1",

    "test_weighted_f1",
]


summary_df = pd.DataFrame({

    "mean":
        results_df[
            metric_columns
        ].mean(),

    "std":
        results_df[
            metric_columns
        ].std(
            ddof=1
        ),
}).T


print(
    "\nThree-seed final test summary:"
)


display(
    summary_df.round(4)
)


summary_df.to_csv(

    FINAL_TEST_DIR
    / "three_seed_test_summary.csv"
)


mean_macro_f1 = (
    results_df[
        "test_macro_f1"
    ]
    .mean()
)


std_macro_f1 = (
    results_df[
        "test_macro_f1"
    ]
    .std(
        ddof=1
    )
)


print(
    "\nFINAL TEST MACRO F1:"
)


print(
    f"{mean_macro_f1:.4f}"
    " ± "
    f"{std_macro_f1:.4f}"
)


# ============================================================
# 26. THREE-SEED PER-CLASS TEST SUMMARY
# ============================================================

per_class_df = pd.DataFrame(
    all_per_class_results
)


assert (
    len(per_class_df)
    == 15
)


per_class_summary = (

    per_class_df

    .groupby(
        [
            "class_id",
            "class_name",
        ],
        as_index=False,
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        support=(
            "support",
            "first"
        ),
    )
)


print(
    "\nThree-seed per-class FINAL TEST summary:"
)


display(
    per_class_summary.round(4)
)


per_class_df.to_csv(

    FINAL_TEST_DIR
    / "three_seed_per_class_raw.csv",

    index=False,
)


per_class_summary.to_csv(

    FINAL_TEST_DIR
    / "three_seed_per_class_summary.csv",

    index=False,
)


# ============================================================
# 27. SAVE FINAL EXPERIMENT METADATA
# ============================================================

metadata = {

    "experiment":
        "QC-filtered 1x augmented XLM-R",

    "model":
        MODEL_NAME,

    "test_file":
        str(
            TEST_PATH
        ),

    "test_samples":
        int(
            len(test_df)
        ),

    "test_class_distribution":
        {
            str(k): int(v)
            for k, v
            in actual_test_counts.items()
        },

    "max_length":
        MAX_LENGTH,

    "test_batch_size":
        TEST_BATCH_SIZE,

    "seeds":
        SEEDS,

    "checkpoints":
        {
            str(seed):
                str(
                    QC_CHECKPOINTS[
                        seed
                    ]
                )
            for seed
            in SEEDS
        },

    "checkpoint_selection":
        (
            "Best checkpoint selected using "
            "validation Macro-F1 only."
        ),

    "test_set_usage":
        (
            "Final evaluation only; "
            "no tuning, model selection, or "
            "checkpoint selection used test results."
        ),

    "mean_test_macro_f1":
        float(
            mean_macro_f1
        ),

    "std_test_macro_f1":
        float(
            std_macro_f1
        ),
}


with open(

    FINAL_TEST_DIR
    / "final_test_metadata.json",

    "w",

    encoding="utf-8",

) as file:

    json.dump(
        metadata,
        file,
        indent=4,
    )


# ============================================================
# 28. FINAL VERIFICATION
# ============================================================

assert (
    results_df[
        "test_samples"
    ]
    == 2003
).all()


assert (
    per_class_df
    .groupby(
        "seed"
    )
    .size()
    .eq(
        5
    )
    .all()
)


print(
    "\n"
    + "=" * 80
)

print(
    "QC-FILTERED 1x FINAL TEST EVALUATION COMPLETE"
)

print("=" * 80)


print(
    "Seeds evaluated:",
    SEEDS
)

print(
    "Test samples per seed:",
    results_df[
        "test_samples"
    ].tolist()
)

print(
    "\nMean test Macro F1:",
    f"{mean_macro_f1:.4f}"
)

print(
    "Test Macro F1 standard deviation:",
    f"{std_macro_f1:.4f}"
)

print(
    "\nAll final QC-filtered test results saved to:"
)

print(
    FINAL_TEST_DIR
)

QC-FILTERED 1x FINAL TEST CHECKPOINTS
Seed 42: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_42/checkpoint-3437
Seed 43: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_43/checkpoint-4419
Seed 44: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_44/checkpoint-1473

All three retained QC-filtered checkpoints verified.

FINAL TEST-SET INTEGRITY CHECK
Test samples: 2003
Missing tweets: 0
Missing labels: 0

Test class distribution:


,label,count
0,0,481
1,1,1070
2,2,156
3,3,168
4,4,128



Test-set integrity check: PASSED


Map:   0%|          | 0/2003 [00:00<?, ? examples/s]


Test DataLoader samples: 2003
Test inference batch size: 4

Final test results directory:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented_results/final_test


QC-FILTERED 1x FINAL TEST - SEED 42
Evaluation device: cuda


...


Three-seed final test summary:


,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1
mean,0.7064,0.5939,0.6425,0.6081,0.6987
std,0.0196,0.0208,0.0279,0.0244,0.0247



FINAL TEST MACRO F1:
0.6081 ± 0.0244

Three-seed per-class FINAL TEST summary:


,class_id,class_name,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support
0,0,Abusive/Offensive,0.5998,0.0458,0.3936,0.0609,0.4748,0.0592,481.0
1,1,Normal,0.8365,0.0252,0.8717,0.0046,0.8536,0.0127,1070.0
2,2,Religious Hate,0.5263,0.0220,0.7329,0.0386,0.6124,0.0238,156.0
3,3,Sexism,0.4985,0.0209,0.5813,0.0431,0.5359,0.0157,168.0
4,4,Profane,0.5083,0.0046,0.6328,0.0271,0.5636,0.0135,128.0



QC-FILTERED 1x FINAL TEST EVALUATION COMPLETE
Seeds evaluated: [42, 43, 44]
Test samples per seed: [2003, 2003, 2003]

Mean test Macro F1: 0.6081
Test Macro F1 standard deviation: 0.0244

All final QC-filtered test results saved to:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented_results/final_test


In [35]:
# ============================================================
# FIND THE ACTUAL QC-FILTERED CHECKPOINT PATHS
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

print(
    "Current notebook working directory:"
)

print(
    Path.cwd()
)


print("\nSearching for QC-filtered checkpoints...")


expected_names = {
    42: "checkpoint-3437",
    43: "checkpoint-4419",
    44: "checkpoint-1473",
}


FOUND_QC_CHECKPOINTS = {}


for seed, checkpoint_name in expected_names.items():

    matches = list(
        PROJECT_ROOT.rglob(
            checkpoint_name
        )
    )

    # Keep only directories belonging to the final 1x experiment.
    matches = [
        path
        for path in matches
        if (
            path.is_dir()
            and
            "xlm_roberta_final_1x_augmented"
            in str(path)
        )
    ]

    print(
        f"\nSeed {seed}:"
    )

    for path in matches:

        print(
            path
        )

    assert len(matches) == 1, (
        f"Expected exactly one checkpoint "
        f"for seed {seed}, found {len(matches)}."
    )

    FOUND_QC_CHECKPOINTS[
        seed
    ] = matches[0]


print("\n" + "=" * 70)

print(
    "ACTUAL QC CHECKPOINTS FOUND"
)

print("=" * 70)


for seed, path in (
    FOUND_QC_CHECKPOINTS.items()
):

    print(
        f"Seed {seed}:",
        path
    )

Current notebook working directory:
/home/jovyan/project work/data_analyssis

Searching for QC-filtered checkpoints...

Seed 42:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_42/checkpoint-3437

Seed 43:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_43/checkpoint-4419

Seed 44:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_44/checkpoint-1473

ACTUAL QC CHECKPOINTS FOUND
Seed 42: /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_42/checkpoint-3437
Seed 43: /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_43/checkpoint-4419
Seed 44: /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_final_1x_augmented/seed_44/checkpoint-1473
